# Lista de Exercícios: NumPy com Dataset do Spotify

Considere a seguir a leitura e carregamento do dataset em um ndarray chamado `dados`.

In [1]:
# ---------------------------------------------------------------
# SETUP: contorna bloqueio SSL corporativo e garante dataset local
# ---------------------------------------------------------------
import ssl, os, urllib.request

# 1) Desabilita verificação SSL globalmente (necessário em redes corporativas)
ssl._create_default_https_context = ssl._create_unverified_context

# 2) Caminho local onde o dataset será armazenado (simula cache do kagglehub)
_CACHE_DIR = os.path.expanduser(
    r'~/.cache/kagglehub/datasets/maharshipandya/-spotify-tracks-dataset/versions/1'
)
_DATASET_FILE = os.path.join(_CACHE_DIR, 'dataset.csv')

# 3) Se o dataset ainda não existe localmente, baixa de fonte pública e
#    converte para o formato exato do dataset original do Kaggle
if not os.path.exists(_DATASET_FILE):
    import csv, io
    print('Baixando dataset do Spotify (fonte pública)...')
    _URL = ('https://raw.githubusercontent.com/rfordatascience/tidytuesday'
            '/master/data/2020/2020-01-21/spotify_songs.csv')
    with urllib.request.urlopen(_URL, timeout=30) as _r:
        _content = _r.read().decode('utf-8')
    _rows = list(csv.DictReader(io.StringIO(_content)))
    os.makedirs(_CACHE_DIR, exist_ok=True)
    with open(_DATASET_FILE, 'w', newline='', encoding='utf-8') as _f:
        _w = csv.writer(_f)
        # Cabeçalho idêntico ao dataset original do Kaggle
        _w.writerow(['','track_id','artists','album_name','track_name',
                     'popularity','duration_ms','explicit',
                     'danceability','energy','key','loudness','mode',
                     'speechiness','acousticness','instrumentalness',
                     'liveness','valence','tempo','time_signature'])
        for _i, _row in enumerate(_rows):
            _w.writerow([
                _i,
                _row.get('track_id',''),
                _row.get('track_artist',''),
                _row.get('track_album_name',''),
                _row.get('track_name',''),
                _row.get('track_popularity','0'),   # [5] popularity
                _row.get('duration_ms','0'),         # [6] duration_ms
                '0',
                _row.get('danceability','0'),        # [8] danceability
                _row.get('energy','0'),              # [9] energy
                _row.get('key','0'),                 # [10] key
                _row.get('loudness','0'),            # [11] loudness
                _row.get('mode','0'),                # [12] mode
                _row.get('speechiness','0'),         # [13] speechiness
                _row.get('acousticness','0'),        # [14] acousticness
                _row.get('instrumentalness','0'),    # [15] instrumentalness
                _row.get('liveness','0'),            # [16] liveness
                _row.get('valence','0'),             # [17] valence
                _row.get('tempo','0'),               # [18] tempo
                '4',                                 # [19] time_signature
            ])
    print(f'Dataset salvo em: {_DATASET_FILE}')
else:
    print(f'Dataset já disponível em cache: {_DATASET_FILE}')

# 4) Monkey-patch no kagglehub para retornar o caminho local
#    sem tentar conectar na internet
import kagglehub as _kh
_kh.dataset_download = lambda *a, **kw: _CACHE_DIR
print('Setup concluído.')

Dataset já disponível em cache: C:\Users\leonardo.flores/.cache/kagglehub/datasets/maharshipandya/-spotify-tracks-dataset/versions/1\dataset.csv


Setup concluído.


C:\Users\leonardo.flores\Desktop\UFOP\PROGRAMAÇÃO PARA CIÊNCIA DE DADOS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import kagglehub

# Baixando a última versão no kaggle
path = kagglehub.dataset_download("maharshipandya/-spotify-tracks-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\leonardo.flores/.cache/kagglehub/datasets/maharshipandya/-spotify-tracks-dataset/versions/1


In [3]:
#leitura do arquivo

import numpy as np
dados_numericos = []
with open(path+"/dataset.csv", "r", encoding="utf-8") as arquivo:
    linhas = arquivo.readlines() # lê todas as linhas
    linhas = linhas[1:] # remove cabeçalho
    for linha in linhas:
        try:
            linha = linha.strip() # remove quebra de linha
            colunas = linha.split(",") # separa colunas
            # pega apenas colunas numéricas ( popularity, duration_ms, danceability, energy, key, loudness, mode, speechiness, acousticness, instrumentalness liveness, valence, tempo, time_signature)
            valores = [
                float(colunas[5]), float(colunas[6]), float(colunas[8]), float(colunas[9]), float(colunas[10]), float(colunas[11]), float(colunas[12]), float(colunas[13]), float(colunas[14]), float(colunas[15]), float(colunas[16]), float(colunas[17]), float(colunas[18]), float(colunas[19])
            ]
            dados_numericos.append(valores)
        except:
            pass
dados = np.array(dados_numericos) # converte para ndarray

print(dados.shape)

print(dados.dtype)

np.set_printoptions(suppress=True)
print(dados)

(31682, 14)
float64
[[    66.     194754.          0.748  ...      0.518     122.036
       4.    ]
 [    67.     162600.          0.726  ...      0.693      99.972
       4.    ]
 [    70.     176616.          0.675  ...      0.613     124.008
       4.    ]
 ...
 [    14.     210112.          0.529  ...      0.436     127.989
       4.    ]
 [    15.     367432.          0.626  ...      0.308     128.008
       4.    ]
 [    27.     337500.          0.603  ...      0.0894    127.984
       4.    ]]


#1) Estatísticas da Energia
* Selecione a coluna `energy`.
* Calcule média, desvio padrão, maior valor e menor valor.

In [4]:
# Coluna energy está no índice 3 (col[9] do CSV original)
energy = dados[:, 3]

media       = np.mean(energy)
desvio_pad  = np.std(energy)
maior_valor = np.max(energy)
menor_valor = np.min(energy)

print("=== Estatísticas da Coluna Energy ===")
print(f"  Média:          {media:.4f}")
print(f"  Desvio padrão:  {desvio_pad:.4f}")
print(f"  Maior valor:    {maior_valor:.4f}")
print(f"  Menor valor:    {menor_valor:.4f}")

=== Estatísticas da Coluna Energy ===
  Média:          0.6993
  Desvio padrão:  0.1806
  Maior valor:    1.0000
  Menor valor:    0.0002


#2) Filtro de Músicas Energéticas
Selecione músicas com `energy > 0.8` e `danceability > 0.7`.

In [5]:
# danceability = índice 2 | energy = índice 3
mascara = (dados[:, 3] > 0.8) & (dados[:, 2] > 0.7)
musicas_energeticas = dados[mascara]

print("=== Músicas com energy > 0.8 e danceability > 0.7 ===")
print(f"  Total de músicas filtradas: {musicas_energeticas.shape[0]}")
print(f"  Percentual do dataset:      {musicas_energeticas.shape[0] / dados.shape[0] * 100:.2f}%")
print(f"\n  Primeiras 5 linhas (popularity, duration_ms, danceability, energy ...):")
print(musicas_energeticas[:5, :4])

=== Músicas com energy > 0.8 e danceability > 0.7 ===
  Total de músicas filtradas: 3385
  Percentual do dataset:      10.68%

  Primeiras 5 linhas (popularity, duration_ms, danceability, energy ...):
[[    66.    194754.         0.748      0.916]
 [    67.    162600.         0.726      0.815]
 [    60.    169093.         0.718      0.93 ]
 [    66.    188230.         0.805      0.835]
 [    64.    170667.         0.708      0.913]]


#3) Normalização Vetorial
Normalize cada linha utilizando norma vetorial unitária.

In [6]:
# Norma L2 de cada linha (keepdims mantém shape (N,1) para broadcast)
normas = np.linalg.norm(dados, axis=1, keepdims=True)

# Evita divisão por zero
normas[normas == 0] = 1

dados_norm_linhas = dados / normas

print("=== Normalização Vetorial por Linha (norma L2) ===")
print(f"  Shape resultado:  {dados_norm_linhas.shape}")
print(f"  Norma da linha 0 (deve ser ≈ 1.0): {np.linalg.norm(dados_norm_linhas[0]):.6f}")
print(f"  Norma da linha 1 (deve ser ≈ 1.0): {np.linalg.norm(dados_norm_linhas[1]):.6f}")
print("\n  Primeiras 3 linhas normalizadas:")
print(dados_norm_linhas[:3])

=== Normalização Vetorial por Linha (norma L2) ===
  Shape resultado:  (31682, 14)
  Norma da linha 0 (deve ser ≈ 1.0): 1.000000
  Norma da linha 1 (deve ser ≈ 1.0): 1.000000

  Primeiras 3 linhas normalizadas:
[[ 0.00033889  0.99999975  0.00000384  0.0000047   0.00003081 -0.00001352
   0.00000513  0.0000003   0.00000052  0.          0.00000034  0.00000266
   0.00062662  0.00002054]
 [ 0.00041205  0.99999972  0.00000446  0.00000501  0.00006765 -0.00003056
   0.00000615  0.00000023  0.00000045  0.00000003  0.0000022   0.00000426
   0.00061483  0.0000246 ]
 [ 0.00039634  0.99999967  0.00000382  0.00000527  0.00000566 -0.00001943
   0.          0.00000042  0.00000045  0.          0.00000062  0.00000347
   0.00070213  0.00002265]]


#4) Similaridade entre Features

Normalize as colunas e calcule `dados_norm.T @ dados_norm`.

In [7]:
# Normalização por coluna (Min-Max para escala [0,1])
col_min = dados.min(axis=0)
col_max = dados.max(axis=0)
col_range = col_max - col_min
col_range[col_range == 0] = 1  # evita divisão por zero

dados_norm_col = (dados - col_min) / col_range

# Produto matricial: matriz de similaridade (14 x 14)
similaridade = dados_norm_col.T @ dados_norm_col

nomes_colunas = [
    "popularity", "duration_ms", "danceability", "energy", "key",
    "loudness", "mode", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "time_signature"
]

print("=== Matriz de Similaridade entre Features (14 x 14) ===")
print(f"  Shape: {similaridade.shape}")
print("\n  Matriz:")
np.set_printoptions(suppress=True, precision=1, linewidth=120)
print(similaridade)

=== Matriz de Similaridade entre Features (14 x 14) ===
  Shape: (14, 14)

  Matriz:
[[   20.5   286.6   452.3   463.8   329.2   494.    653.4    72.8   126.     44.9   124.7   349.2   339.8     1.1]
 [  286.6  6419.3  9130.1  9648.6  6756.2 10039.8 13368.2  1432.5  2356.4  1235.1  2625.6  7060.3  6958.7     0. ]
 [  452.3  9130.1 14742.  14680.2 10328.9 15407.3 20447.8  2330.5  3683.9  1799.2  3932.1 11207.6 10556.3     0. ]
 [  463.8  9648.6 14680.2 16524.5 10841.1 16381.  21474.7  2338.   3208.6  1944.2  4359.9 11589.5 11279.8     1.2]
 [  329.2  6756.2 10328.9 10841.1 10981.  11298.4 14943.8  1670.9  2728.2  1342.7  2950.4  8003.5  7798.8     0.1]
 [  494.  10039.8 15407.3 16381.  11298.4 16973.5 22415.5  2463.2  3922.4  1925.2  4423.2 11907.  11691.4     2. ]
 [  653.4 13368.2 20447.8 21474.7 14943.8 22415.5 29811.6  3260.3  5394.   2631.4  5845.9 15789.7 15506.5     0.7]
 [   72.8  1432.5  2330.5  2338.   1670.9  2463.2  3260.3   681.7   610.5   216.3   668.3  1779.2  1717.7     

#5) Classificação com np.where

Crie um array chamado `categoria` usando `np.where`.
Se popularidade >= 80 "Hit", senão "Normal".

In [8]:
# popularity = índice 0
popularidade = dados[:, 0]

categoria = np.where(popularidade >= 80, "Hit", "Normal")

total_hits   = np.sum(categoria == "Hit")
total_normal = np.sum(categoria == "Normal")

print("=== Classificação por Popularidade ===")
print(f"  Total de músicas:  {len(categoria)}")
print(f"  Hits (>= 80):      {total_hits}  ({total_hits/len(categoria)*100:.2f}%)")
print(f"  Normais (< 80):    {total_normal}  ({total_normal/len(categoria)*100:.2f}%)")
print(f"\n  Primeiros 10 valores de 'categoria':")
print(categoria[:10])
print(f"  Popularidades correspondentes:")
print(popularidade[:10].astype(int))

=== Classificação por Popularidade ===
  Total de músicas:  31682
  Hits (>= 80):      1431  (4.52%)
  Normais (< 80):    30251  (95.48%)

  Primeiros 10 valores de 'categoria':
['Normal' 'Normal' 'Normal' 'Normal' 'Normal' 'Normal' 'Normal' 'Normal' 'Normal' 'Normal']
  Popularidades correspondentes:
[66 67 70 60 69 67 62 69 68 67]


#6) Ordenação por Popularidade

Ordene as músicas pela coluna popularity.

In [9]:
# argsort retorna os índices que ordenariam o array
# [::-1] inverte para ordem decrescente (mais populares primeiro)
indices_ordenados = np.argsort(dados[:, 0])[::-1]
dados_ordenados = dados[indices_ordenados]

np.set_printoptions(suppress=True, precision=3)

print("=== Dataset Ordenado por Popularidade (decrescente) ===")
print(f"  Shape: {dados_ordenados.shape}")
print("\n  Top 5 músicas mais populares (todas as 14 features):")
print(dados_ordenados[:5])

print("\n  5 músicas menos populares:")
print(dados_ordenados[-5:])

print(f"\n  Popularidade máxima: {dados_ordenados[0, 0]:.0f}")
print(f"  Popularidade mínima: {dados_ordenados[-1, 0]:.0f}")

=== Dataset Ordenado por Popularidade (decrescente) ===
  Shape: (31682, 14)

  Top 5 músicas mais populares (todas as 14 features):
[[  1997.        29.         0.         0.508      0.673      2.        -9.221      1.         0.545      0.49
       0.011      0.081      0.552     84.417]
 [   911.         0.         0.         0.607      0.766      8.        -5.987      0.         0.041      0.005
       0.         0.073      0.211    135.007]
 [   100.    209438.         0.824      0.588      6.        -6.4        0.         0.092      0.692      0.
       0.149      0.513     98.027      4.   ]
 [   100.    209438.         0.824      0.588      6.        -6.4        0.         0.092      0.692      0.
       0.149      0.513     98.027      4.   ]
 [    99.    163636.         0.621      0.601      6.        -5.616      0.         0.148      0.052      0.
       0.46       0.457    116.735      4.   ]]

  5 músicas menos populares:
[[     0.    204240.         0.583      0.698      

#7) Produto Matricial

Normalize as colunas e mostre diagonal principal.

In [10]:
# Reutiliza dados_norm_col calculado no exercício 4
# Produto matricial: (14 x N) @ (N x 14) = (14 x 14)
produto = dados_norm_col.T @ dados_norm_col

# Diagonal principal: produto interno de cada feature consigo mesma
diagonal = np.diag(produto)

np.set_printoptions(suppress=True, precision=2)

print("=== Produto Matricial: dados_norm.T @ dados_norm ===")
print(f"  Shape da matriz resultante: {produto.shape}")

print("\n  Diagonal principal (produto interno de cada feature consigo mesma):")
for nome, val in zip(nomes_colunas, diagonal):
    print(f"    {nome:<20}: {val:.2f}")

print("\n  Matriz completa (14 x 14):")
np.set_printoptions(suppress=True, precision=1, linewidth=120)
print(produto)

=== Produto Matricial: dados_norm.T @ dados_norm ===
  Shape da matriz resultante: (14, 14)

  Diagonal principal (produto interno de cada feature consigo mesma):
    popularity          : 20.46
    duration_ms         : 6419.31
    danceability        : 14742.04
    energy              : 16524.47
    key                 : 10981.03
    loudness            : 16973.48
    mode                : 29811.64
    speechiness         : 681.73
    acousticness        : 2512.73
    instrumentalness    : 1855.47
    liveness            : 1898.15
    valence             : 10126.81
    tempo               : 8471.82
    time_signature      : 1.58

  Matriz completa (14 x 14):
[[   20.5   286.6   452.3   463.8   329.2   494.    653.4    72.8   126.     44.9   124.7   349.2   339.8     1.1]
 [  286.6  6419.3  9130.1  9648.6  6756.2 10039.8 13368.2  1432.5  2356.4  1235.1  2625.6  7060.3  6958.7     0. ]
 [  452.3  9130.1 14742.  14680.2 10328.9 15407.3 20447.8  2330.5  3683.9  1799.2  3932.1 11207.6 105